In the previous IBM notebooks, Collaborative Filtering was done with KNN. KNN looks at a user, finds the 5 users who have similar rating histories, and averages their ratings for the target movie.

**Matrix Factorization** approaches the problem completely differently. It asks: "What if we could represent every user and every movie as a small vector of numbers (an embedding), and predict the rating by taking their dot product?"

Predicted Rating$_{u,i} = v_{user} * v_{item}$


<h2>The "Cold Start" Problem</h2>

In common PhD literature, you will constantly see this scenario when trying to recommend brand new movies.

<h3>Collaborative Filtering</h3> 
(whether User-Based, Item-Based, or SVD) is entirely blind to metadata. It cannot read the title, and it does not know the genre. Its entire universe is bounded by the `user-item` interaction matrix.If a movie has zero ratings, its column in that mathematical matrix is completely blank (or filled with zeros/nulls).
<br>

**Deconstructing Item-Based vs. User-Based CF**
To understand why Item-Based CF fails here, we have to look at exactly how both memory-based approaches calculate "similarity."

**1. User-Based CF** 
The Goal: Find similar users.
The Math: It compares rows in the matrix. User $A$ and User $B$ are similar if they gave similar ratings to the exact same overlapping set of movies.**The Recommendation:** If User $A$ is similar to User $B$, recommend movies User $B$ liked that User $A$ hasn't seen.

**2. Item-Based CF** (The IBM Module's other method)The Goal: Find similar items.The Math: It compares columns in the matrix. Item $X$ and Item $Y$ are similar if they were rated similarly by the exact same overlapping group of users.The Recommendation: If you liked Item $X$, the system recommends Item $Y$, because the crowd of users who liked $X$ also tended to like $Y$.

**Why the New Movie Fails:** Look closely at the math for Item-Based CF. Two items are only similar if they share a common audience of users.If our brand-new movie has zero ratings, it shares zero users with any other movie in the database. When the algorithm tries to calculate the cosine similarity between the new movie's blank vector and "The Matrix's" highly populated vector, the mathematical result is $0$. The model literally cannot see the new movie to compare it to anything.

This is the golden rule of Collaborative Filtering: **Similarity is defined exclusively by shared human interaction**. No interactions, no similarity.

I shall use the Python library called Surprise (pip install scikit-surprise). It is built specifically for Recommender Systems and makes implementing MF algorithms (like SVD) very clean.

First Task:
Before I can train any model, I have to prepare the data. The Surprise library doesn't take raw pandas DataFrames directly. It requires the data to be in a very specific format: a structure containing only three columns: `userID`, `itemID`, and `rating`.

In [1]:
from surprise import Dataset, SVD, Reader
from surprise.accuracy import rmse
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Loading the datasets

movie_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/BxZuF3FrO7Bdw6McwsBaBw/movies.csv')
rating_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/R-bYYyyf7s3IUE5rsssmMw/ratings.csv')
tag_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/UZKHhXSl7Ft7t9mfUFZJPQ/tags.csv')

In [3]:
rating_df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [4]:
rating_df.isna().any()

userId       False
movieId      False
rating       False
timestamp    False
dtype: bool

In [5]:
rating_df.drop(columns='timestamp', inplace=True)

In [6]:
rating_df.rename(columns={'userId':'userID', 'movieId':'itemID'}, inplace=True)

In [7]:
rating_df.head()

,userID,itemID,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0


In [8]:
# A reader is still needed but only the rating_scale param is required.
min_r, max_r = rating_df.rating.min(),rating_df.rating.max() 

reader = Reader(rating_scale=(min_r, max_r))

In [9]:
# The columns must correspond to user id, item id and ratings (in that order).
data = Dataset.load_from_df(rating_df[["userID", "itemID", "rating"]], reader)

In [10]:
type(data)

surprise.dataset.DatasetAutoFolds

**Splitting the dataset into train, test, split**

In [11]:
from surprise.model_selection import train_test_split

train_set, test_set = train_test_split(data, test_size=0.25, random_state=42)

In [12]:
# let's see what type of objects are the train and test sets:

print(type(train_set))
print(type(test_set))

<class 'surprise.trainset.Trainset'>
<class 'list'>


In [13]:
# The test set is a list of tuples, 

test_set[:5]

[(50, 4282, 3.5),
 (603, 2993, 3.0),
 (140, 11, 4.0),
 (262, 497, 4.0),
 (492, 1363, 4.0)]

The train_set, is a different object...
The train_set is not a pandas dataframe, and it is not a list. It is a highly optimized C-level structure (a Cython object) designed specifically for matrix operations.

When Surprise creates the Trainset, it does something crucial: It translates your raw IDs into `"Inner IDs"`.If your pandas dataframe had a User ID of "User_99" and an Item ID of "Movie_ABC", Surprise maps them to dense, continuous integers starting at 0 (e.g., Inner User 0, Inner Item 0). This is required because matrix factorization involves creating an $N \times M$ matrix, and you can't have string indices or gaps in your matrix coordinates.

In [14]:
# 1. Look at the basic dimensions
print(f"Total Users: {train_set.n_users}")
print(f"Total Items: {train_set.n_items}")
print(f"Total Ratings: {train_set.n_ratings}")

# 2. See the mapping dictionaries (Raw ID -> Inner ID)
# Let's peek at the first 5 User mappings
user_mapping = list(train_set._raw2inner_id_users.items())[:5]
print(f"\nFirst 5 User mappings (Raw -> Inner): {user_mapping}")

# 3. Look at the actual ratings
# The ratings are stored as a generator of tuples: (inner_uid, inner_iid, rating)
# Let's pull the first 5 ratings from the generator
ratings_generator = train_set.all_ratings()
first_5_ratings = [next(ratings_generator) for _ in range(5)]
print(f"\nFirst 5 internal ratings (Inner UID, Inner IID, Rating): {first_5_ratings}")

Total Users: 610
Total Items: 8731
Total Ratings: 75627

First 5 User mappings (Raw -> Inner): [(432, 0), (288, 1), (599, 2), (42, 3), (75, 4)]

First 5 internal ratings (Inner UID, Inner IID, Rating): [(0, 0, 4.5), (0, 398, 3.0), (0, 572, 4.0), (0, 501, 2.5), (0, 1348, 3.5)]


**Creating and Training the SVD model**

In [15]:
model = SVD()
model.fit(train_set)

Under the hood, `model.fit(train_set)` has just done gradient descent and created latent embeddings for all the users and movies in the dataset.

**Testing and Evaluation**

Now that the model is trained, I need to know if it's actually any good, testing with the 25% saved data.

I will generate predictions for all the user-item pairs in the test_set and then calculate the Root Mean Square Error (RMSE).

In [16]:
predictions = model.test(test_set)

In [17]:
RMSE = rmse(predictions, verbose=True)

RMSE: 0.8806


**Exercise:**

Find the top 5 movie recommendtaions for userID 1, these must be movies the user has not yet seen and rated.

In [18]:
# get a list of movieIDs not seen by userID 1

x = rating_df[rating_df['userID'] != 1]['itemID'].to_list()

# Convert to a set, to remove duplicates
x = list(set(x))

print(len(x))

9723


In [19]:
# Make this a dataframe instead of iterating

k = pd.DataFrame(x, columns=['movieID'])
k.head()

,movieID
0,1
1,2
2,3
3,4
4,5


Create a function that we can apply to get the predicted ratings

In [20]:
def predict_ratings(movieID, user=1, model=model):
    pred = model.predict(user, movieID)
    return round(float(pred.est), 2)
    

In [21]:
# Applying the ratings to each movie prediction

k['ratings'] = k.apply(lambda x: predict_ratings(x['movieID']), axis=1)

In [22]:
k.head()

,movieID,ratings
0,1,4.51
1,2,4.26
2,3,4.06
3,4,3.89
4,5,3.71


Now to find the top 10 most recommended movies for User 1...


In [23]:
top_movies = k.sort_values(by='ratings', ascending=False)
top_movies = top_movies.head(10)

In [24]:
top_movies = list(top_movies.movieID)
top_movies

[527, 904, 922, 908, 910, 912, 1172, 1136, 1196, 58559]

In [25]:
movie_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [26]:
top_movies = movie_df[movie_df['movieId'].isin(top_movies)]
top_movies

,movieId,title,genres
461,527,Schindler's List (1993),Drama|War
686,904,Rear Window (1954),Mystery|Thriller
690,908,North by Northwest (1959),Action|Adventure|Mystery|Romance|Thriller
692,910,Some Like It Hot (1959),Comedy|Crime
694,912,Casablanca (1942),Drama|Romance
704,922,Sunset Blvd. (a.k.a. Sunset Boulevard) (1950),Drama|Film-Noir|Romance
863,1136,Monty Python and the Holy Grail (1975),Adventure|Comedy|Fantasy
878,1172,Cinema Paradiso (Nuovo cinema Paradiso) (1989),Drama
898,1196,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi
6710,58559,"Dark Knight, The (2008)",Action|Crime|Drama|IMAX


## User-Based Collaborative Filtering Pipeline

I will execute the pure User-Based Collaborative Filtering pipeline. Using the surprise library:

* Calculate the Neighborhood: The algorithm will compute the cosine similarity between User 1 and every other user in the dataset based on their overlapping ratings.

* Find the Unseen: I will ask the algorithm to look at movies User 1 has not rated yet.

* Predict via Neighbors: For each unseen movie, the algorithm will find the most similar users to User 1 who have rated that movie. It will average their ratings (weighted by how similar they are to User 1) to predict what User 1 would rate it.

* Recommend: I will sort those predicted ratings from highest to lowest and present the top movies to User 1.

**Step One: Calculate the Neighbourhood**

In [27]:
from surprise import KNNBasic

# 1. Define the similarity options:
sim_options = {
    'name': 'cosine',
    'user_based': True  
}

# 2. Instantiate the algorithm
knn_model = KNNBasic(sim_options=sim_options)

# 3. Train it on your existing Surprise train_set
knn_model.fit(train_set)

Computing the cosine similarity matrix...
Done computing similarity matrix.


**What Just Happened**?

When I ran fit(), Surprise doesn't do gradient descent. Instead, it literally computes the cosine angle between every single user's rating history and stores those similarity scores in memory.

To visualize it, imagine a giant grid where the rows are Users and the columns are Movies.
* The Vector: User 1's "vector" is their entire row across all movies.
* The Calculation: The algorithm takes User 1's row and User 2's row and calculates the cosine angle between them.
* The Storage: It saves that single similarity score, then moves to User 1 and User 3, computes it, saves it, and so on until it has compared every user to every other user.
<br>
It holds this entire $N \times N$ (User by User) similarity matrix in my computer's RAM.However, there is one brilliant quirk in how Surprise (and Collaborative Filtering in general) calculates this: It only calculates similarity on the overlap.If User 1 has rated a thousand movies, and User 2 has only rated five movies, the algorithm does not calculate the cosine similarity across all 1,000 dimensions. It isolates only the specific movies both users have rated.
<br>
If they both only watched "The Matrix" and "Toy Story," their similarity score is based entirely on those two data points.This leads to a famous structural problem in recommender systems:Because it has to calculate every pair and store it in memory, Memory-Based CF is incredibly fast to train on small datasets, but it physically cannot scale to the size of Netflix or Amazon. If you have 100 million users, trying to store a 100M x 100M similarity matrix in memory will instantly crash your server. This is why huge tech companies generally prefer Model-Based approaches (like SVD) or deep learning!

**Step Two: Find the Unseen**

In [28]:
# 1. Get a set of EVERY unique movie ID in the entire dataframe
all_movies = set(rating_df['itemID'])

# 2. Get a set of ONLY the movie IDs that User 1 has rated
user1_movies = set(rating_df[rating_df['userID'] == 1]['itemID'])

# 3. Mathematically subtract User 1's movies from the global pool
unseen_movies = list(all_movies - user1_movies)
len(unseen_movies)

9492

In [29]:
# create a dataframe of these unseen movies

k = pd.DataFrame(unseen_movies, columns=['unseen_movies'])
k.head()

,unseen_movies
0,2
1,4
2,5
3,7
4,8


In [30]:
# for each movie,add the ave predicted rating of users most similar to user 1 

k['pred_ratings'] = k.apply(
    lambda x: predict_ratings(x.unseen_movies, user=1, model=knn_model)
, axis=1)

In [31]:
k.head()

,unseen_movies,pred_ratings
0,2,3.45
1,4,2.59
2,5,3.08
3,7,3.16
4,8,3.15


**What just happened?**

Inside the `predict()` execution:

Let's say the code asks for a prediction for `Movie ID 99`. Here is exactly what the predict() method does under the hood in a fraction of a second:
* It checks the similarity matrix and isolates User 1's most similar peers.
* It filters that list down to a maximum of $40$ users who have an actual rating recorded for Movie ID 99.
* It computes the weighted average formula using their actual ratings and their similarity scores:$$\frac{\sum (\text{similarity} \cdot \text{rating})}{\sum \text{similarity}}$$
* It spits out the final predicted score (e.g., 4.2) and hands it back to your pandas lambda function.
* The model re-evaluates that 40-person neighborhood for every single movie the lambda function passes to it. The "bunch of similar users" is dynamic — it changes slightly for every prediction depending on who actually watched the movie in question.

The 40-user is a population sze threshold. A default hyperparameter in the KNNbasic model:

* $k = 40$: This is the maximum neighborhood size. For every single prediction the apply function runs, the algorithm scans the similarity matrix in memory and grabs the 40 users most similar to User 1 who have also rated that specific movie.
* $min\_k = 1$: This is the absolute minimum threshold. If fewer than 1 neighbor has rated the movie, the algorithm abandons the Collaborative Filtering calculation for that specific item and simply outputs the global average rating of all movies.

In [32]:
# Top 10 movies to recommend to user 1, based on similar users...

k.sort_values(by='pred_ratings', ascending=False, inplace=True)
top_10 = k.head(10)

In [33]:
top_10

,unseen_movies,pred_ratings
5996,8580,5.0
3784,4788,5.0
7294,115727,5.0
9452,98083,5.0
9476,130970,5.0
5333,7071,5.0
5356,7096,5.0
5378,7122,5.0
5377,7121,5.0
9262,96832,5.0


In [34]:
final_rec = top_10.merge(movie_df, 
             left_on='unseen_movies', 
             right_on='movieId', 
             how='left')

final_rec

,unseen_movies,pred_ratings,movieId,title,genres
0,8580,5.0,8580,Into the Woods (1991),Adventure|Comedy|Fantasy|Musical
1,4788,5.0,4788,Moscow Does Not Believe in Tears (Moskva sleza...,Drama|Romance
2,115727,5.0,115727,Crippled Avengers (Can que) (Return of the 5 D...,Action|Adventure
3,98083,5.0,98083,Jackass 3.5 (2011),Comedy|Documentary
4,130970,5.0,130970,George Carlin: Life Is Worth Losing (2005),Comedy
5,7071,5.0,7071,"Woman Under the Influence, A (1974)",Drama
6,7096,5.0,7096,Rivers and Tides (2001),Documentary
7,7122,5.0,7122,King of Hearts (1966),Comedy|Drama|War
8,7121,5.0,7121,Adam's Rib (1949),Comedy|Romance
9,96832,5.0,96832,Holy Motors (2012),Drama|Fantasy|Musical|Mystery|Sci-Fi


## Mission Accomplished: The CF Foundation
Take a step back and think about the journey sofar. I now have two functioning, fundamentally different recommender models in my toolkit:

* **SVD (Model-Based CF)**: Learned abstract patterns across the whole dataset via gradient descent.

* **KNNBasic (Memory-Based CF)**: Relied strictly on the literal overlapping behavior of User 1's closest historical neighbors.